In [1]:
import torch
import os
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning
import pytorchvideo.data
import torch.utils.data
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    Normalize,
    RandomShortSideScale,
    RemoveKey,
    ShortSideScale,
    UniformTemporalSubsample
)
import torchvision
from torchvision.transforms import (
    Compose,
    Lambda,
    RandomCrop,
    RandomHorizontalFlip,
    CenterCrop
)
import torch.nn as nn
import torch.nn.functional as F

c:\Users\nikam\ML2\Handgesten\.venv\Lib\site-packages\torchvision\transforms\functional_tensor.py:5: UserWarning: The torchvision.transforms.functional_tensor module is deprecated in 0.15 and will be **removed in 0.17**. Please don't rely on it. You probably just need to use APIs in torchvision.transforms.functional or in torchvision.transforms.v2.functional.
  warnings.warn(


In [2]:
class KineticsDataModule(pytorch_lightning.LightningDataModule):

  # Dataset configuration
  _DATA_PATH = 'new_data/'
  _CLIP_DURATION = 3  # Duration of sampled clip for each video
  _BATCH_SIZE =2
  _NUM_WORKERS = 0# Number of parallel processes fetching data

  def train_dataloader(self):
    """
    Create the Kinetics train partition from the list of video labels
    in {self._DATA_PATH}/train.csv. Add transform that subsamples and
    normalizes the video before applying the scale, crop and flip augmentations.
    """
    train_transform = Compose(
        [
        ApplyTransformToKey(
          key="video",
          transform=Compose(
              [
                UniformTemporalSubsample(10),
                Lambda(lambda x: x / 255.0),
                Normalize((0.45, 0.45, 0.45), (0.225, 0.225, 0.225)),
                RandomShortSideScale(min_size=256, max_size=320),
                RandomCrop(244),
                RandomHorizontalFlip(p=0.5),
              ]
            ),
          ),
        ]
    )
    train_dataset = pytorchvideo.data.Kinetics(
        data_path=os.path.join("train.csv"),
        clip_sampler=pytorchvideo.data.make_clip_sampler("random", self._CLIP_DURATION),
        transform=train_transform
    )
    return torch.utils.data.DataLoader(
        train_dataset,
        batch_size=self._BATCH_SIZE,
        num_workers=self._NUM_WORKERS,
    )

  def val_dataloader(self):
    """
    Create the Kinetics validation partition from the list of video labels
    in {self._DATA_PATH}/val
    """
    val_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(10),
            Lambda(lambda x: x / 255.0),
            Normalize((0.45, 0.45, 0.45), (0.225, 0.225, 0.225)),
            ShortSideScale(256),
            CenterCrop(244),
        ]),
    ),
])
    val_dataset = pytorchvideo.data.Kinetics(
        data_path=os.path.join("val.csv"),
        clip_sampler=pytorchvideo.data.make_clip_sampler("uniform", self._CLIP_DURATION),
        transform=val_transform,
        decode_audio=False,
    )
    return torch.utils.data.DataLoader(
        val_dataset,
        batch_size=self._BATCH_SIZE,
        num_workers=self._NUM_WORKERS,
    )

In [3]:
data_module = KineticsDataModule()

In [6]:
import cv2
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [ ]:
class HangestenDatensatz(Dataset):
    
    def __init__(self, data_path, df, label_map, custom_transform):
        self.data = data_path
        self.df = list(zip(df['path'], df['label']))
        self.label_map = label_map
        self.custom_transform = custom_transform
        self.num_frames = 30
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        video_path, labels = self.df[index]
        video_tensor = self.read(video_path, transform=self.custom_transform)
        label = self.label_map[labels]

        return video_tensor, torch.tensor(label, dtype=torch.long)

    def read(self, video, transform):
        frames = []
        cap = cv2.VideoCapture(filename=video)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, total_frames-1, self.num_frames, dtype=int)

        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if not ret:
                print("error with video reading")
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            frames.append(transform(frame))
        cap.release()

        return torch.stack(frames)




In [8]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

In [ ]:
class mobilenet_handgesten(nn.Module):

    def __init__(self, num_classes: int = 10, hidden_dim: int = 128,
                    num_layers: int = 2, freeze_backbone: bool = True):
            super().__init__()

            backbone = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
            # Drop the classification head; keep only the conv feature extractor
            self.feature_extractor = backbone.features   # outputs (B, 576, h, w)
            self.avgpool = nn.AdaptiveAvgPool2d(1)
            feature_dim = 576  # MobileNetV3-Small final channel count

            if freeze_backbone:
                for param in self.feature_extractor.parameters():
                    param.requires_grad = False

            self.lstm = nn.LSTM(
                input_size=feature_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=0.3,
                bidirectional=True,
            )
            self.classifier = nn.Sequential(
                nn.Dropout(0.5),
                nn.Linear(hidden_dim * 2, num_classes),  # *2 for bidirectional
            )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)                          # merge batch & time
        feats = self.avgpool(self.feature_extractor(x))      # (B*T, 576, 1, 1)
        feats = feats.flatten(1)                             # (B*T, 576)
        feats = feats.view(B, T, -1)                         # (B, T, 576)
        out, _ = self.lstm(feats)                            # (B, T, hidden*2)
        return self.classifier(out[:, -1, :])                # last timestep